# 05 — Cost Regression

**Objective** — compare regression models with grouped cross-validation, then evaluate the selected model on the untouched test set.

**Input** — cleaned analytical dataset and training-only preprocessing.

**Output** — selected regression model and final test metrics.

## 1. Prepare the Grouped Split

In [ ]:
from pathlib import Path
import sys
import pandas as pd

current_directory = Path.cwd().resolve()
project_root = next(
    directory for directory in [current_directory, *current_directory.parents]
    if (directory / 'src').is_dir() and (directory / 'notebooks').is_dir()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.features import prepare_feature_frame, build_preprocessor
from src.modeling import grouped_train_test_split, get_regression_candidates, compare_models_cv
from src.evaluation import regression_metrics

processed_data_path = project_root / 'data' / 'processed' / 'analytical_dataset.csv'
analytical_dataset = pd.read_csv(processed_data_path)
features, target_charges, customer_groups = prepare_feature_frame(analytical_dataset)
training_features, test_features, training_charges, test_charges, training_groups, test_groups = grouped_train_test_split(
    features, target_charges, customer_groups
)
preprocessor = build_preprocessor(training_features)

## 2. Compare Candidate Models

In [ ]:
regression_models = get_regression_candidates(preprocessor)
regression_comparison = compare_models_cv(
    regression_models, training_features, training_charges, training_groups, 'regression'
)
regression_comparison

## 3. Evaluate the Selected Model

In [ ]:
selected_model_name = regression_comparison.iloc[0]['model']
selected_regression_model = regression_models[selected_model_name].fit(training_features, training_charges)
regression_predictions = selected_regression_model.predict(test_features)
{'selected_model': selected_model_name, **regression_metrics(test_charges, regression_predictions)}

## Takeaways

Model selection uses grouped cross-validation inside the training set only. MAE is the main error metric. RMSE and R² give extra information about large errors and explained variation.